# Simulasi Sistem Dinamik Ketersediaan Gula Nasional
**UTS Pemodelan dan Simulasi** | Produksi, konsumsi, dan ketergantungan impor dengan skenario peningkatan produktivitas, rendemen, dan perluasan areal tebu

**Alur notebook:** data → kalibrasi → validasi (MAPE + uji kondisi ekstrem) → skenario → sensitivitas → deploy web (Streamlit)

**Struktur model (stock-flow, Euler dt = 0,25 tahun)**

| Stok | Persamaan |
|---|---|
| Luas areal tebu A | dA/dt = g_exp·A + D/τ_dev − g_conv·A |
| Lahan dalam pengembangan D | dD/dt = program_perluasan − D/τ_dev |
| Stok gula nasional S | dS/dt = produksi + impor − konsumsi |
| Penduduk P | dP/dt = g_pop·P |

Variabel bantu: produksi = A · produktivitas tebu · rendemen; konsumsi = P · c(t) dengan c(t) menuju titik jenuh; impor = max(0, konsumsi − produksi + (S_target − S)/τ_s); SSR = produksi / konsumsi.


In [ ]:
!pip -q install scipy matplotlib pandas streamlit

## 1. Data
Sumber: Outlook Tebu 2025 (Pusdatin Kementan), BPS, dan berita yang mengutip BPS. **Verifikasi kolom impor 2017 (interpolasi) dan jumlah penduduk (pembulatan) ke tabel BPS sebelum dikumpulkan.**

In [ ]:
%%writefile data_gula.csv
tahun,luas_panen_ha,produksi_ton,impor_ton,penduduk_juta,catatan
2014,477122,2579173,2933823,254.9,
2015,445650,2497998,3360000,258.2,
2016,440733,2204619,4746047,261.1,
2017,425617,2121671,4890000,264.2,impor 2017 = interpolasi (ganti dengan data BPS)
2018,414846,2170947,5028000,267.0,
2019,411435,2227046,4090000,269.6,
2020,420505,2130719,5539000,270.2,
2021,447399,2350809,5482000,272.7,
2022,488982,2405907,6007000,275.8,
2023,504776,2271009,5069455,278.7,
2024,520823,2465514,5313529,281.6,angka sementara

In [ ]:
%%writefile sd_model.py
"""
Model System Dynamics: Ketersediaan Gula Nasional Indonesia
Stok  : Luas areal tebu (A), Lahan dalam pengembangan (D), Stok gula nasional (S), Penduduk (P)
Aliran: ekspansi organik, alih fungsi, pematangan lahan baru, produksi, impor, konsumsi
Sumber data: Outlook Tebu 2025 (Pusdatin Kementan), BPS, berita yang mengutip BPS.
"""
import numpy as np
import pandas as pd
from scipy.optimize import least_squares, curve_fit

# ---------------------------------------------------------------- DATA
def load_data(path="data_gula.csv"):
    df = pd.read_csv(path)
    df["konsumsi_ton"] = df["produksi_ton"] + df["impor_ton"]          # apparent consumption
    df["kons_per_kapita_kg"] = df["konsumsi_ton"] / (df["penduduk_juta"] * 1000)
    df["SSR_aktual"] = df["produksi_ton"] / df["konsumsi_ton"]
    return df

# ---------------------------------------------------------------- SKENARIO
SKENARIO = {
    "Baseline (tren saat ini)":      dict(),
    "Peningkatan produktivitas":     dict(y_target=85.0),
    "Peningkatan rendemen":          dict(r_target=0.10),
    "Perluasan areal (+350 rb ha)":  dict(ha_program=350_000),
    "Kombinasi tiga intervensi":     dict(y_target=85.0, r_target=0.10, ha_program=350_000),
    "Roadmap Perpres 40/2023 penuh": dict(y_target=93.0, r_target=0.112, ha_program=700_000),
}

# ---------------------------------------------------------------- MODEL
def simulate(p, skenario=None, t0=2014, t1=2035, dt=0.25, mulai=2026, selesai=2030):
    """Simulasi Euler. Mengembalikan DataFrame tahunan."""
    s = skenario or {}
    y0, r0 = p["y_tebu0"], p["r_eff"]
    y_t = s.get("y_target", y0)
    r_t = s.get("r_target", r0)
    ha_total = s.get("ha_program", 0.0) * s.get("realisasi", 1.0)

    A, D, P = p["A0"], 0.0, p["pop0"]
    S = p["cover"] * (P * p["c0"] * 1000)           # stok awal = stok target
    rows, n_steps = [], int(round((t1 - t0) / dt))
    for k in range(n_steps + 1):
        t = t0 + k * dt
        frac = float(np.clip((t - mulai) / (selesai - mulai), 0, 1))
        y_tebu = y0 + (y_t - y0) * frac
        rendemen = r0 + (r_t - r0) * frac
        in_program = (ha_total / (selesai - mulai)) if mulai <= t < selesai else 0.0

        produksi = A * y_tebu * rendemen
        c_kapita = p["c_inf"] - (p["c_inf"] - p["c0"]) * np.exp(-p["k_c"] * (t - t0))   # menuju titik jenuh
        konsumsi = P * c_kapita * 1000
        s_target = p["cover"] * konsumsi
        impor = max(0.0, konsumsi - produksi + (s_target - S) / p["tau_s"])

        if abs(t - round(t)) < 1e-9:
            rows.append(dict(tahun=int(round(t)), luas_ha=A, lahan_pengembangan_ha=D,
                             y_tebu=y_tebu, rendemen=rendemen, produksi_ton=produksi,
                             penduduk_juta=P, konsumsi_ton=konsumsi, impor_ton=impor,
                             stok_ton=S, SSR=produksi / konsumsi,
                             IDR=impor / (produksi + impor) if (produksi + impor) > 0 else 0))
        # laju perubahan
        matang = D / p["tau_dev"]
        if t < p["tahun_rezim"]:
            g_exp = p["g_exp"]
        elif t < p["tahun_proyeksi"]:
            g_exp = p["g_exp2"]
        else:
            g_exp = p["g_exp_proy"]
        dA = g_exp * A + matang - p["g_conv"] * A
        dD = in_program - matang
        dS = produksi + impor - konsumsi
        dP = p["g_pop"] * P
        A += dA * dt; D += dD * dt; S += dS * dt; P += dP * dt
    return pd.DataFrame(rows)

# ---------------------------------------------------------------- KALIBRASI
def kalibrasi(df, g_conv=0.02, y_tebu0=70.0, cover=0.15, tau_s=1.0, tau_dev=1.0,
              tahun_rezim=2021, tahun_proyeksi=2024, net_growth_proyeksi=0.0207):
    """Kalibrasi parameter. net_growth_proyeksi = rata-rata pertumbuhan luas panen 2016-2025
    (Outlook Tebu 2025: 2,07%/tahun) dipakai untuk tahun >= tahun_proyeksi."""
    t = df.tahun.values - df.tahun.values[0]
    g_pop = np.polyfit(t, np.log(df.penduduk_juta.values), 1)[0]
    f = lambda tt, c0, cinf, k: cinf - (cinf - c0) * np.exp(-k * tt)
    (c0, c_inf, k_c), _ = curve_fit(f, t, df.kons_per_kapita_kg.values, p0=[22, 30, 0.3],
                                    bounds=([10, 20, 0.01], [40, 60, 2.0]))
    p = dict(A0=float(df.luas_panen_ha.iloc[0]), pop0=float(df.penduduk_juta.iloc[0]),
             c0=float(c0), c_inf=float(c_inf), k_c=float(k_c), g_pop=float(g_pop),
             g_conv=g_conv, y_tebu0=y_tebu0, cover=cover, tau_s=tau_s, tau_dev=tau_dev,
             tahun_rezim=tahun_rezim, tahun_proyeksi=tahun_proyeksi,
             g_exp=0.03, g_exp2=0.05, r_eff=0.07, g_exp_proy=net_growth_proyeksi + g_conv)
    t0, t1 = int(df.tahun.min()), int(df.tahun.max())

    def resid(x):
        q = dict(p, g_exp=x[0], g_exp2=x[1], r_eff=x[2])
        sim = simulate(q, t0=t0, t1=t1).set_index("tahun")
        a = (sim.luas_ha.values - df.luas_panen_ha.values) / df.luas_panen_ha.values
        b = (sim.produksi_ton.values - df.produksi_ton.values) / df.produksi_ton.values
        return np.r_[a, b]

    sol = least_squares(resid, x0=[0.0, 0.05, 0.07], bounds=([-0.1, -0.1, 0.03], [0.2, 0.3, 0.12]))
    p["g_exp"], p["g_exp2"], p["r_eff"] = map(float, sol.x)
    return p

# ---------------------------------------------------------------- VALIDASI
def mape(aktual, sim):
    aktual, sim = np.asarray(aktual, float), np.asarray(sim, float)
    return float(np.mean(np.abs((aktual - sim) / aktual)) * 100)

def validasi(df, p, tahun=None):
    t0, t1 = int(df.tahun.min()), int(df.tahun.max())
    sim = simulate(p, t0=t0, t1=t1).set_index("tahun")
    d = df.set_index("tahun")
    if tahun is not None:
        sim, d = sim.loc[tahun], d.loc[tahun]
    return {
        "Luas areal":   mape(d.luas_panen_ha, sim.luas_ha),
        "Produksi":     mape(d.produksi_ton, sim.produksi_ton),
        "Konsumsi":     mape(d.konsumsi_ton, sim.konsumsi_ton),
        "Impor":        mape(d.impor_ton, sim.impor_ton),
        "SSR":          mape(d.SSR_aktual, sim.SSR),
    }

# ---------------------------------------------------------------- SENSITIVITAS
def sensitivitas(p, tahun=2030, delta=0.10):
    base = simulate(p).set_index("tahun").loc[tahun, "SSR"]
    out = {}
    for nama, key in [("Rendemen efektif", "r_eff"), ("Produktivitas tebu", "y_tebu0"),
                      ("Laju ekspansi lahan (proyeksi)", "g_exp_proy"), ("Laju alih fungsi", "g_conv"),
                      ("Konsumsi/kapita jenuh", "c_inf"), ("Pertumbuhan penduduk", "g_pop")]:
        hasil = []
        for sgn in (-1, +1):
            q = dict(p); q[key] = p[key] * (1 + sgn * delta)
            hasil.append(simulate(q).set_index("tahun").loc[tahun, "SSR"] - base)
        out[nama] = hasil
    return base, out


# ---------------------------------------------------------------- UJI KONDISI EKSTREM
def uji_ekstrem(p):
    """Uji kondisi ekstrem: hasil harus masuk akal secara logika."""
    hasil = {}
    q = dict(p, r_eff=0.0)                                  # rendemen nol -> produksi nol, semua impor
    r = simulate(q).set_index("tahun").loc[2030]
    hasil["Rendemen = 0"] = dict(produksi=r.produksi_ton, SSR=r.SSR, impor_vs_konsumsi=r.impor_ton / r.konsumsi_ton)
    q = dict(p, c0=1e-6, c_inf=1e-6)                        # konsumsi ~ 0 -> impor 0
    r = simulate(q).set_index("tahun").loc[2030]
    hasil["Konsumsi ~ 0"] = dict(impor=r.impor_ton)
    q = dict(p, r_eff=0.5)                                  # rendemen sangat tinggi -> SSR >> 1, impor 0
    r = simulate(q).set_index("tahun").loc[2030]
    hasil["Rendemen sangat tinggi"] = dict(SSR=r.SSR, impor=r.impor_ton)
    return hasil

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sd_model as m
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

df = m.load_data("data_gula.csv")
display(df)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(df.tahun, df.luas_panen_ha/1e3, "o-"); ax[0].set_title("Luas panen (ribu ha)")
ax[1].plot(df.tahun, df.produksi_ton/1e6, "o-", label="Produksi"); ax[1].plot(df.tahun, df.konsumsi_ton/1e6, "s-", label="Konsumsi (apparent)")
ax[1].set_title("Produksi vs konsumsi (juta ton)"); ax[1].legend()
ax[2].plot(df.tahun, df.SSR_aktual*100, "o-"); ax[2].set_title("SSR aktual (%)")
plt.tight_layout(); plt.show()

## 2. Kalibrasi parameter
Penduduk: regresi log-linear. Konsumsi per kapita: kurva menuju titik jenuh (`curve_fit`). Laju ekspansi lahan dan rendemen efektif: *least squares* terhadap luas areal dan produksi 2014-2024. Laju ekspansi dibedakan sebelum dan sesudah 2021 (perubahan rezim kebijakan dan pabrik gula baru). Alih fungsi lahan, produktivitas tebu awal (70 ton/ha), dan stok penyangga adalah asumsi.

In [ ]:
P = m.kalibrasi(df)
pd.Series({k: v for k, v in P.items()}, name="nilai").to_frame()

## 3. Validasi
### 3.1 Behavior reproduction (MAPE)

In [ ]:
val = m.validasi(df, P)
display(pd.DataFrame({"MAPE (%)": val}).round(2))

sv = m.simulate(P, t1=2024)
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
for x, (jd, ak, sm) in zip(ax, [("Luas areal (ha)", df.luas_panen_ha, sv.luas_ha),
                                ("Produksi (ton)", df.produksi_ton, sv.produksi_ton),
                                ("Konsumsi (ton)", df.konsumsi_ton, sv.konsumsi_ton),
                                ("Impor (ton)", df.impor_ton, sv.impor_ton)]):
    x.plot(df.tahun, ak, "ko-", ms=3, label="Aktual"); x.plot(sv.tahun, sm, "r--", label="Simulasi"); x.set_title(jd); x.legend(fontsize=7)
plt.tight_layout(); plt.show()

Kriteria: MAPE < 10% dianggap baik, 10-20% cukup. Pembanding eksternal: Outlook Tebu 2025 memperkirakan produksi 2025 sebesar 2,59 juta ton dan luas panen 526 ribu ha.

In [ ]:
s25 = m.simulate(P).set_index("tahun").loc[2025]
print(f"Model 2025 : produksi {s25.produksi_ton/1e6:.3f} jt ton | luas {s25.luas_ha/1e3:.1f} rb ha")
print("Outlook 2025: produksi 2.589 jt ton | luas 526.4 rb ha")
print(f"Selisih produksi: {(s25.produksi_ton/2589073-1)*100:+.1f}% | selisih luas: {(s25.luas_ha/526445-1)*100:+.1f}%")

### 3.2 Uji kondisi ekstrem
Rendemen = 0 harus membuat produksi = 0 dan seluruh konsumsi dipenuhi impor; konsumsi ≈ 0 harus membuat impor = 0; rendemen sangat tinggi harus membuat impor = 0 dan SSR > 1.

In [ ]:
for k, v in m.uji_ekstrem(P).items():
    print(k, {kk: round(float(vv), 3) for kk, vv in v.items()})

## 4. Skenario kebijakan (intervensi 2026-2030)
- **Produktivitas**: 70 → 85 ton tebu/ha
- **Rendemen**: ±7,2% → 10%
- **Perluasan areal**: +350 ribu ha (50% target Perpres 40/2023, dengan jeda pematangan 1 tahun)
- **Kombinasi** ketiganya
- **Roadmap Perpres 40/2023 penuh**: 93 ton/ha, rendemen 11,2%, +700 ribu ha

Pertumbuhan neto areal baseline = 2,07%/tahun (rata-rata 2016-2025, Outlook Tebu 2025).

In [ ]:
hasil = {}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for nama, sc in m.SKENARIO.items():
    s = m.simulate(P, sc, t1=2035)
    hasil[nama] = s.set_index("tahun")
    ax[0].plot(s.tahun, s.SSR*100, label=nama)
    ax[1].plot(s.tahun, s.produksi_ton/1e6)
    ax[2].plot(s.tahun, s.impor_ton/1e6)
ax[0].axhline(100, color="g", lw=.8); ax[0].set_title("SSR (%)"); ax[0].legend(fontsize=6)
ax[1].plot(hasil["Baseline (tren saat ini)"].index, hasil["Baseline (tren saat ini)"].konsumsi_ton/1e6, "k:", label="Konsumsi")
ax[1].set_title("Produksi (juta ton)"); ax[1].legend(fontsize=7)
ax[2].set_title("Impor (juta ton)")
plt.tight_layout(); plt.show()

tabel = pd.DataFrame({n: {"SSR 2030 (%)": h.loc[2030,"SSR"]*100, "SSR 2035 (%)": h.loc[2035,"SSR"]*100,
                          "Produksi 2030 (jt ton)": h.loc[2030,"produksi_ton"]/1e6,
                          "Impor 2030 (jt ton)": h.loc[2030,"impor_ton"]/1e6,
                          "Luas areal 2030 (rb ha)": h.loc[2030,"luas_ha"]/1e3} for n, h in hasil.items()}).T
display(tabel.round(2))

## 5. Analisis sensitivitas (±10% parameter, dampak ke SSR 2030)

In [ ]:
base_ssr, out = m.sensitivitas(P, tahun=2030, delta=0.10)
nama = list(out); lo = [out[k][0]*100 for k in nama]; hi = [out[k][1]*100 for k in nama]
urut = np.argsort([max(abs(a), abs(b)) for a, b in zip(lo, hi)])
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh([nama[i] for i in urut], [lo[i] for i in urut], label="-10%"); ax.barh([nama[i] for i in urut], [hi[i] for i in urut], label="+10%")
ax.axvline(0, color="k", lw=.8); ax.set_xlabel("Perubahan SSR 2030 (poin persen)"); ax.legend(); plt.tight_layout(); plt.show()
print(f"SSR baseline 2030: {base_ssr*100:.1f}%")

## 6. Deploy aplikasi web (Streamlit)
Cell berikut menulis `app.py` dan `requirements.txt`. Untuk deploy permanen: unggah `app.py`, `sd_model.py`, `data_gula.csv`, dan `requirements.txt` ke repositori GitHub, lalu hubungkan di [share.streamlit.io](https://share.streamlit.io). Untuk uji cepat dari Colab, jalankan cell terakhir (memakai localtunnel).

In [ ]:
%%writefile app.py
import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import sd_model as m

st.set_page_config(page_title="SD Gula Nasional", page_icon="🍬", layout="wide")

@st.cache_data
def siapkan():
    df = m.load_data("data_gula.csv")
    return df, m.kalibrasi(df)

df, P0 = siapkan()

st.title("🍬 Simulasi Sistem Dinamik Ketersediaan Gula Nasional")
st.caption("Produksi, konsumsi, dan ketergantungan impor dengan skenario peningkatan produktivitas, rendemen, dan perluasan areal tebu")

# ------------------------------------------------------------ sidebar
st.sidebar.header("Skenario kustom")
y_t = st.sidebar.slider("Target produktivitas tebu (ton/ha)", 70.0, 100.0, 85.0, 1.0)
r_t = st.sidebar.slider("Target rendemen (%)", 7.0, 12.0, 10.0, 0.1) / 100
ha = st.sidebar.slider("Tambahan areal baru (ribu ha)", 0, 700, 350, 25) * 1000
real = st.sidebar.slider("Realisasi program (%)", 0, 100, 100, 5) / 100
mulai, selesai = st.sidebar.slider("Periode intervensi", 2026, 2035, (2026, 2030))
g_net = st.sidebar.slider("Pertumbuhan neto areal baseline (%/th)", -2.0, 5.0, 2.07, 0.1) / 100
horizon = st.sidebar.slider("Tahun akhir simulasi", 2030, 2040, 2035)

p = dict(P0, g_exp_proy=g_net + P0["g_conv"])
kustom = dict(y_target=y_t, r_target=r_t, ha_program=ha, realisasi=real)
base = m.simulate(p, t1=horizon, mulai=mulai, selesai=selesai)
sim = m.simulate(p, kustom, t1=horizon, mulai=mulai, selesai=selesai)

tab1, tab2, tab3, tab4, tab5 = st.tabs(["Skenario kustom", "Perbandingan skenario", "Validasi model",
                                         "Sensitivitas", "Metode & data"])

# ------------------------------------------------------------ tab 1
with tab1:
    th = st.selectbox("Tahun yang ditinjau", list(range(2026, horizon + 1)), index=min(4, horizon - 2026))
    a, b = base.set_index("tahun").loc[th], sim.set_index("tahun").loc[th]
    c1, c2, c3, c4 = st.columns(4)
    c1.metric(f"SSR {th}", f"{b.SSR*100:.1f}%", f"{(b.SSR-a.SSR)*100:+.1f} poin vs baseline")
    c2.metric(f"Produksi {th}", f"{b.produksi_ton/1e6:.2f} jt ton", f"{(b.produksi_ton-a.produksi_ton)/1e6:+.2f}")
    c3.metric(f"Impor {th}", f"{b.impor_ton/1e6:.2f} jt ton", f"{(b.impor_ton-a.impor_ton)/1e6:+.2f}", delta_color="inverse")
    c4.metric(f"Luas areal {th}", f"{b.luas_ha/1e3:.0f} rb ha", f"{(b.luas_ha-a.luas_ha)/1e3:+.0f}")

    fig, ax = plt.subplots(2, 2, figsize=(11, 6.5))
    ax[0, 0].plot(df.tahun, df.produksi_ton / 1e6, "ko", ms=4, label="Produksi aktual")
    ax[0, 0].plot(base.tahun, base.produksi_ton / 1e6, "--", label="Produksi baseline")
    ax[0, 0].plot(sim.tahun, sim.produksi_ton / 1e6, label="Produksi skenario")
    ax[0, 0].plot(sim.tahun, sim.konsumsi_ton / 1e6, color="tab:red", label="Konsumsi")
    ax[0, 0].set_title("Produksi vs konsumsi (juta ton)"); ax[0, 0].legend(fontsize=7)
    ax[0, 1].plot(df.tahun, df.SSR_aktual * 100, "ko", ms=4, label="SSR aktual")
    ax[0, 1].plot(base.tahun, base.SSR * 100, "--", label="Baseline")
    ax[0, 1].plot(sim.tahun, sim.SSR * 100, label="Skenario")
    ax[0, 1].axhline(100, color="g", lw=0.8); ax[0, 1].set_title("Self-Sufficiency Ratio (%)"); ax[0, 1].legend(fontsize=7)
    ax[1, 0].plot(df.tahun, df.luas_panen_ha / 1e3, "ko", ms=4)
    ax[1, 0].plot(base.tahun, base.luas_ha / 1e3, "--"); ax[1, 0].plot(sim.tahun, sim.luas_ha / 1e3)
    ax[1, 0].set_title("Luas areal tebu (ribu ha)")
    ax[1, 1].plot(df.tahun, df.impor_ton / 1e6, "ko", ms=4)
    ax[1, 1].plot(base.tahun, base.impor_ton / 1e6, "--"); ax[1, 1].plot(sim.tahun, sim.impor_ton / 1e6)
    ax[1, 1].set_title("Impor gula (juta ton)")
    for x in ax.flat: x.grid(alpha=0.3)
    plt.tight_layout(); st.pyplot(fig)
    st.dataframe(sim[["tahun", "luas_ha", "y_tebu", "rendemen", "produksi_ton", "konsumsi_ton", "impor_ton", "SSR", "IDR"]]
                 .round(3), width="stretch", hide_index=True)

# ------------------------------------------------------------ tab 2
with tab2:
    rows = []
    for nama, sc in m.SKENARIO.items():
        s = m.simulate(p, sc, t1=2035).set_index("tahun")
        rows.append({"Skenario": nama, "SSR 2030 (%)": s.loc[2030, "SSR"] * 100,
                     "SSR 2035 (%)": s.loc[2035, "SSR"] * 100,
                     "Produksi 2030 (jt ton)": s.loc[2030, "produksi_ton"] / 1e6,
                     "Impor 2030 (jt ton)": s.loc[2030, "impor_ton"] / 1e6})
    tb = pd.DataFrame(rows).round(2)
    st.dataframe(tb, width="stretch", hide_index=True)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for nama, sc in m.SKENARIO.items():
        s = m.simulate(p, sc, t1=2035)
        ax.plot(s.tahun, s.SSR * 100, label=nama)
    ax.axhline(100, color="g", lw=0.8); ax.set_ylabel("SSR (%)"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
    st.pyplot(fig)

# ------------------------------------------------------------ tab 3
with tab3:
    val = m.validasi(df, P0)
    st.subheader("MAPE model terhadap data 2014–2024")
    st.dataframe(pd.DataFrame({"Variabel": val.keys(), "MAPE (%)": [round(v, 2) for v in val.values()]}),
                 hide_index=True)
    sv = m.simulate(P0, t1=2024)
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
    for x, (jd, ak, sm) in zip(ax, [("Luas areal (ha)", df.luas_panen_ha, sv.luas_ha),
                                    ("Produksi (ton)", df.produksi_ton, sv.produksi_ton),
                                    ("Impor (ton)", df.impor_ton, sv.impor_ton)]):
        x.plot(df.tahun, ak, "ko-", ms=3, label="Aktual"); x.plot(sv.tahun, sm, "r--", label="Simulasi")
        x.set_title(jd); x.grid(alpha=0.3); x.legend(fontsize=7)
    plt.tight_layout(); st.pyplot(fig)
    st.write("**Uji kondisi ekstrem (tahun 2030)**")
    st.json({k: {kk: float(vv) for kk, vv in v.items()} for k, v in m.uji_ekstrem(P0).items()})
    st.write("**Parameter hasil kalibrasi**")
    st.json({k: (round(v, 5) if isinstance(v, float) else v) for k, v in P0.items()})

# ------------------------------------------------------------ tab 4
with tab4:
    th = st.selectbox("Tahun SSR", [2030, 2035], index=0)
    delta = st.slider("Perubahan parameter ±(%)", 5, 30, 10, 5) / 100
    base_ssr, out = m.sensitivitas(p, tahun=th, delta=delta)
    st.write(f"SSR baseline {th}: **{base_ssr*100:.1f}%**")
    nama = list(out.keys()); lo = [out[k][0] * 100 for k in nama]; hi = [out[k][1] * 100 for k in nama]
    urut = np.argsort([max(abs(l), abs(h)) for l, h in zip(lo, hi)])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh([nama[i] for i in urut], [lo[i] for i in urut], color="tab:blue", label=f"-{int(delta*100)}%")
    ax.barh([nama[i] for i in urut], [hi[i] for i in urut], color="tab:orange", label=f"+{int(delta*100)}%")
    ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("Perubahan SSR (poin persen)"); ax.legend(); ax.grid(alpha=0.3)
    st.pyplot(fig)

# ------------------------------------------------------------ tab 5
with tab5:
    st.markdown("""
**Struktur model (stock-flow)**
- Stok: luas areal tebu *A*, lahan dalam pengembangan *D* (delay 1 tahun), stok gula nasional *S*, penduduk *P*.
- Produksi = *A* × produktivitas tebu × rendemen efektif.
- Konsumsi = *P* × konsumsi per kapita (menuju titik jenuh).
- Impor = max(0, konsumsi − produksi + (stok target − *S*)/τ) → loop penyeimbang.
- SSR = produksi / konsumsi. Konsumsi = produksi + impor (apparent consumption, definisi Outlook Tebu).

**Sumber data:** Outlook Tebu 2025 (Pusdatin Kementan; data Ditjen Perkebunan), BPS, dan berita yang mengutip BPS.
Impor 2017 dan jumlah penduduk perlu diverifikasi dengan tabel BPS (lihat kolom *catatan* pada data).
""")
    st.dataframe(df, width="stretch", hide_index=True)

In [ ]:
%%writefile requirements.txt
streamlit>=1.50
numpy
pandas
scipy
matplotlib

In [ ]:
# Uji cepat di Colab: jalankan, lalu buka URL localtunnel (password = IP yang dicetak)
!npm install -q localtunnel
!streamlit run app.py &>/content/streamlit.log &
!curl -s https://loca.lt/mytunnelpassword; echo
!npx localtunnel --port 8501